# Condition Occurrence — Allscripts Sunrise (SCM)

OMOP CDM v5.4 `condition_occurrence` hydration from the structured SCM warehouse diagnosis views provided by Northwell.

## Source Views
- `_exponent._bronze_allscripts_scm.vw_diagnosis` — diagnosis fact rows with `ClientVisitGUID`, diagnosis type, comments, and `dbo_scadiagnosis` columns.
- `_exponent._bronze_allscripts_scm.vw_diagdim` — diagnosis dimension/code lookup for ICD-9-CM, ICD-10-CM, and SNOMED values.

## Strategy
- Use `vw_diagnosis.ClientVisitGUID` to resolve `visit_occurrence_id` and `person_id` through the existing SCM visit mapping.
- Join `vw_diagdim` dynamically on the shared diagnosis-dimension key.
- Prefer SNOMED, then ICD-10-CM, then ICD-9-CM source codes when multiple code systems are present.
- Map source codes to OMOP standard Condition concepts through `concept` + `concept_relationship`.
- Filter in this notebook to standard concepts in the Condition domain because the client view is intentionally unfiltered.

## Dependencies
- SCM person and visit_occurrence hydration must run first.
- `_exponent.omop_mapping.source_to_visit_occurrence` must contain active SCM rows keyed from `dbo_cv3clientvisit.GUID`.
- OMOP vocabulary tables (`concept`, `concept_relationship`) must be loaded.

## Full Refresh
Reset cells are left commented. For a production full reload, run the three-step reset first: truncate SCM gold, delete SCM silver rows, and delete SCM `source_to_condition_occurrence` rows.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
from pyspark.sql import functions as F


In [0]:
source = "allscripts_scm"
diagnosis_view = "_exponent._bronze_allscripts_scm.vw_diagnosis"
diagdim_view = "_exponent._bronze_allscripts_scm.vw_diagdim"

print(f"Source system : {source}")
print(f"Diagnosis view: {diagnosis_view}")
print(f"Diag dim view : {diagdim_view}")

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_scm.condition_occurrence;


In [0]:
%sql
-- DELETE FROM _exponent.omop_silver.condition_occurrence
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- DELETE FROM _exponent.omop_mapping.source_to_condition_occurrence
-- WHERE source_system = 'allscripts_scm';


In [0]:
# Source view inspection. Run this first after the VPN catalog is attached.
for view_name in [diagnosis_view, diagdim_view]:
    print(f"Schema for {view_name}")
    display(spark.sql(f"DESCRIBE TABLE {view_name}"))
    print(f"Sample rows from {view_name}")
    display(spark.table(view_name).limit(10))

In [0]:
# Build silver staging from the client-provided SCM diagnosis views.
# The SCM views were delivered outside this repo, so this cell discovers the
# usable code columns and diagnosis-dimension relationship at runtime.

from pyspark.sql import functions as F


def _column_map(table_name):
    return {c.lower(): c for c in spark.table(table_name).columns}


def _pick(cols, candidates, label, required=False):
    for candidate in candidates:
        actual = cols.get(candidate.lower())
        if actual:
            return actual
    if required:
        raise ValueError(
            f"Could not find {label}. Tried: {', '.join(candidates)}. "
            "Run the source inspection cell and add the actual column name."
        )
    return None


def _escape_identifier(col):
    return col.replace("`", "``")


def _q(alias, col):
    return f"{alias}.`{_escape_identifier(col)}`"


def _spark_col(col):
    return F.col(f"`{_escape_identifier(col)}`")


def _coalesce(alias, cols, candidates, cast_type=None):
    picked = [_pick(cols, [candidate], candidate) for candidate in candidates]
    picked = [col for col in picked if col]
    if not picked:
        return "NULL"
    exprs = [_q(alias, col) for col in picked]
    if cast_type:
        exprs = [f"TRY_CAST({expr} AS {cast_type})" for expr in exprs]
    return "COALESCE(" + ", ".join(exprs) + ")"


def _contains_pick(cols, include_terms, exclude_terms=None):
    exclude_terms = exclude_terms or []
    matches = []
    for lower_name, actual in cols.items():
        compact = lower_name.replace("_", "")
        if all(term in compact for term in include_terms) and not any(term in compact for term in exclude_terms):
            matches.append(actual)
    return matches


def _join_key_token(col_name):
    token = col_name.lower().replace("_", "")
    token = token.replace("diagnosis", "diag")
    token = token.replace("dimension", "dim")
    token = token.replace("dictionary", "dict")
    return token


def _is_join_candidate(col_name):
    token = _join_key_token(col_name)
    excluded = [
        "visit", "clientvisit", "patient", "person", "client", "type", "status",
        "user", "provider", "employee", "facility", "location", "date", "time",
        "created", "updated", "modified", "entered", "comment", "sequence",
    ]
    return (token.endswith("id") or token.endswith("guid")) and not any(term in token for term in excluded)


def _candidate_join_columns(cols):
    preferred = []
    fallback = []
    for actual in cols.values():
        token = _join_key_token(actual)
        if token in {"visitid", "clientvisitguid", "diagtypedimid"}:
            continue
        if _is_join_candidate(actual):
            if "diag" in token or "icd" in token or "snomed" in token or "dict" in token:
                preferred.append(actual)
            else:
                fallback.append(actual)
    return preferred + fallback


def _find_join_key(left_cols, right_cols):
    exact_candidates = [
        "DiagDimID", "DiagnosisDimID", "DiagnosisDimensionID", "DiagnosisIDDim",
        "DiagnosisCodeDimID", "DiagCodeDimID", "DiagID", "DiagGUID", "DiagnosisGUID",
        "DiagDictDimID", "DiagnosisDictDimID", "ProblemDimID", "TermDimID",
    ]
    for candidate in exact_candidates:
        if candidate.lower() in left_cols and candidate.lower() in right_cols:
            return left_cols[candidate.lower()], right_cols[candidate.lower()], "exact column match"

    explicit_pairs = [
        ("DiagnosisDimID", "DiagDimID"),
        ("DiagnosisDimensionID", "DiagDimID"),
        ("DiagnosisCodeDimID", "DiagDimID"),
        ("DiagCodeDimID", "DiagDimID"),
        ("DiagID", "DiagDimID"),
        ("DiagnosisID", "DiagDimID"),
        ("DiagnosisID", "DiagnosisDimID"),
        ("DiagDictDimID", "DiagDimID"),
        ("DiagnosisDictDimID", "DiagDimID"),
        ("ProblemDimID", "DiagDimID"),
        ("TermDimID", "DiagDimID"),
    ]
    for left, right in explicit_pairs:
        if left.lower() in left_cols and right.lower() in right_cols:
            return left_cols[left.lower()], right_cols[right.lower()], "known diagnosis dimension pair"

    right_by_token = {_join_key_token(actual): actual for actual in right_cols.values()}
    for actual in left_cols.values():
        token = _join_key_token(actual)
        if token in right_by_token and _is_join_candidate(actual):
            return actual, right_by_token[token], "normalized key match"

    return None


def _profile_join_values(table_name, candidates, sample_limit=50000):
    profiles = []
    for col in candidates[:20]:
        df = (
            spark.table(table_name)
            .select(_spark_col(col).cast("string").alias("join_value"))
            .where(_spark_col(col).isNotNull())
            .distinct()
            .limit(sample_limit)
            .cache()
        )
        count = df.count()
        if count > 0:
            profiles.append((col, df, count))
        else:
            df.unpersist()
    return profiles


def _score_join_candidates(left_candidates, right_candidates, sample_limit=50000):
    scores = []
    left_profiles = _profile_join_values(diagnosis_view, left_candidates, sample_limit)
    right_profiles = _profile_join_values(diagdim_view, right_candidates, sample_limit)
    try:
        for left_col, left_df, left_count in left_profiles:
            for right_col, right_df, right_count in right_profiles:
                overlap = left_df.join(right_df, "join_value", "inner").count()
                if overlap > 0:
                    scores.append((left_col, right_col, overlap, left_count, right_count, overlap / max(left_count, 1)))
    finally:
        for _, df, _ in left_profiles + right_profiles:
            df.unpersist()
    return sorted(scores, key=lambda row: (row[5], row[2]), reverse=True)


def _code_expr_from_candidates(alias, cols, exact_candidates, contains_groups):
    picked = [_pick(cols, [candidate], candidate) for candidate in exact_candidates]
    for include_terms, exclude_terms in contains_groups:
        picked.extend(_contains_pick(cols, include_terms, exclude_terms))
    unique = []
    seen = set()
    for col in picked:
        if col and col.lower() not in seen:
            seen.add(col.lower())
            unique.append(col)
    if not unique:
        return "NULL", []
    exprs = [f"TRY_CAST({_q(alias, col)} AS STRING)" for col in unique]
    cleaned = [f"NULLIF(TRIM({expr}), '')" for expr in exprs]
    return "COALESCE(" + ", ".join(cleaned) + ")", unique


def _combine_expr(*exprs):
    exprs = [expr for expr in exprs if expr and expr != "NULL"]
    if not exprs:
        return "NULL"
    if len(exprs) == 1:
        return exprs[0]
    return "COALESCE(" + ", ".join(exprs) + ")"


diag_cols = _column_map(diagnosis_view)
dim_cols = _column_map(diagdim_view)

client_visit_guid_col = _pick(diag_cols, ["ClientVisitGUID"], "ClientVisitGUID", required=True)
patient_source_col = _pick(
    diag_cols,
    ["PatientDimID", "ClientGUID", "ClientGuid", "PatientGUID", "PatientGuid", "ClientID", "PatientID"],
    "patient/client source id",
)
diag_type_col = _pick(diag_cols, ["dtd_DiagType", "DiagType", "DiagnosisType"], "diagnosis type")
diag_comment_col = _pick(diag_cols, ["dtd_Comment", "Comment", "DiagnosisComment"], "diagnosis comment")
source_id_col = _pick(
    diag_cols,
    ["DiagnosisID", "SCADiagnosisID", "ScaDiagnosisID", "DiagID", "DiagnosisGUID", "GUID", "ID"],
    "diagnosis row id",
)

diagnosis_ts_expr = _coalesce(
    "d",
    diag_cols,
    [
        "DiagnosisDtm", "DiagnosisDate", "DiagDtm", "DiagDate", "DiagnosedDtm", "DiagnosedDate",
        "OnsetDtm", "OnsetDate", "AdmitDtm", "ServiceDtm", "ServiceDate", "AuthoredDtm",
        "CreatedWhen", "CreatedDtm", "CreateDtm", "CreatedDate", "Entered", "EnteredDtm",
        "LastModifiedDtm", "UpdatedDtm", "UpdateDtm",
    ],
    "TIMESTAMP",
)

snomed_d_expr, snomed_d_cols = _code_expr_from_candidates(
    "d",
    diag_cols,
    ["SnomedCode", "SNOMEDCode", "SNOMEDCTCode", "SnomedCTCode", "SNOMED", "Snomed", "SNOMED_CODE", "SNOMEDCT_CODE", "SNOMEDConceptCode", "SnomedConceptCode"],
    [(["snomed"], ["desc", "description", "name", "text"]), (["snomed", "code"], [])],
)
icd10_d_expr, icd10_d_cols = _code_expr_from_candidates(
    "d",
    diag_cols,
    ["ICD10DiagnosisCode", "ICD10DiagnosisCODE", "ICD10Code", "ICD10CMCode", "ICD10", "ICD10CM", "ICD10_CODE", "ICD10CM_CODE", "DiagnosisICD10Code", "DiagICD10Code"],
    [(["icd10"], ["desc", "description", "name", "text"]), (["icd", "10"], ["desc", "description", "name", "text"])],
)
icd9_d_expr, icd9_d_cols = _code_expr_from_candidates(
    "d",
    diag_cols,
    ["ICD9DiagnosisCode", "ICD9DiagnosisCODE", "ICD9Code", "ICD9CMCode", "ICD9", "ICD9CM", "ICD9_CODE", "ICD9CM_CODE", "DiagnosisICD9Code", "DiagICD9Code"],
    [(["icd9"], ["desc", "description", "name", "text"]), (["icd", "9"], ["desc", "description", "name", "text"])],
)

snomed_dd_expr, snomed_dd_cols = _code_expr_from_candidates(
    "dd",
    dim_cols,
    ["SnomedCode", "SNOMEDCode", "SNOMEDCTCode", "SnomedCTCode", "SNOMED", "Snomed", "SNOMED_CODE", "SNOMEDCT_CODE", "SNOMEDConceptCode", "SnomedConceptCode"],
    [(["snomed"], ["desc", "description", "name", "text"]), (["snomed", "code"], [])],
)
icd10_dd_expr, icd10_dd_cols = _code_expr_from_candidates(
    "dd",
    dim_cols,
    ["ICD10DiagnosisCode", "ICD10DiagnosisCODE", "ICD10Code", "ICD10CMCode", "ICD10", "ICD10CM", "ICD10_CODE", "ICD10CM_CODE", "DiagnosisICD10Code", "DiagICD10Code"],
    [(["icd10"], ["desc", "description", "name", "text"]), (["icd", "10"], ["desc", "description", "name", "text"])],
)
icd9_dd_expr, icd9_dd_cols = _code_expr_from_candidates(
    "dd",
    dim_cols,
    ["ICD9DiagnosisCode", "ICD9DiagnosisCODE", "ICD9Code", "ICD9CMCode", "ICD9", "ICD9CM", "ICD9_CODE", "ICD9CM_CODE", "DiagnosisICD9Code", "DiagICD9Code"],
    [(["icd9"], ["desc", "description", "name", "text"]), (["icd", "9"], ["desc", "description", "name", "text"])],
)

join_key = _find_join_key(diag_cols, dim_cols)
join_scores = []
if not join_key and (snomed_dd_cols or icd10_dd_cols or icd9_dd_cols):
    left_candidates = _candidate_join_columns(diag_cols)
    right_candidates = _candidate_join_columns(dim_cols)
    print(f"Candidate vw_diagnosis join columns: {left_candidates}")
    print(f"Candidate vw_diagdim join columns: {right_candidates}")
    join_scores = _score_join_candidates(left_candidates, right_candidates)
    if join_scores:
        left, right, overlap, left_count, right_count, ratio = join_scores[0]
        join_key = (left, right, f"sampled data overlap: {overlap} shared distinct values; diagnosis distinct={left_count}; diagdim distinct={right_count}; ratio={ratio:.4f}")

use_diagdim = bool(join_key)
if use_diagdim:
    snomed_expr = _combine_expr(snomed_d_expr, snomed_dd_expr)
    icd10_expr = _combine_expr(icd10_d_expr, icd10_dd_expr)
    icd9_expr = _combine_expr(icd9_d_expr, icd9_dd_expr)
    source_join_sql = f"""
  FROM {diagnosis_view} d
  LEFT JOIN {diagdim_view} dd
    ON CAST({_q('d', join_key[0])} AS STRING) = CAST({_q('dd', join_key[1])} AS STRING)
"""
else:
    snomed_expr = snomed_d_expr
    icd10_expr = icd10_d_expr
    icd9_expr = icd9_d_expr
    source_join_sql = f"""
  FROM {diagnosis_view} d
"""

print(f"vw_diagnosis columns: {sorted(diag_cols.values())}")
print(f"vw_diagdim columns: {sorted(dim_cols.values())}")
print(f"Detected diagnosis dimension join: {join_key}")
print(f"Detected diagnosis-view SNOMED columns: {snomed_d_cols}")
print(f"Detected diagnosis-view ICD10 columns : {icd10_d_cols}")
print(f"Detected diagnosis-view ICD9 columns  : {icd9_d_cols}")
print(f"Detected diagdim SNOMED columns       : {snomed_dd_cols}")
print(f"Detected diagdim ICD10 columns        : {icd10_dd_cols}")
print(f"Detected diagdim ICD9 columns         : {icd9_dd_cols}")
if join_scores:
    print("Top diagnosis dimension join scores:")
    for row in join_scores[:10]:
        print(row)

if snomed_expr == "NULL" and icd10_expr == "NULL" and icd9_expr == "NULL":
    raise ValueError(
        "Could not find usable SNOMED, ICD10, or ICD9 columns on vw_diagnosis, "
        "and vw_diagdim could not be joined or had no detected code columns. "
        f"vw_diagnosis columns: {sorted(diag_cols.values())}; "
        f"vw_diagdim columns: {sorted(dim_cols.values())}; "
        f"sampled join scores: {join_scores[:10]}"
    )

if not use_diagdim and (snomed_dd_cols or icd10_dd_cols or icd9_dd_cols):
    print("WARNING: vw_diagdim has diagnosis code columns, but no defensible join key was detected. Building from vw_diagnosis code columns only.")

code_expr = f"COALESCE({snomed_expr}, {icd10_expr}, {icd9_expr})"
vocab_expr = f"""
CASE
  WHEN {snomed_expr} IS NOT NULL THEN 'SNOMED'
  WHEN {icd10_expr} IS NOT NULL THEN 'ICD10CM'
  WHEN {icd9_expr} IS NOT NULL THEN 'ICD9CM'
  ELSE NULL
END
"""

if source_id_col:
    source_id_expr = f"CAST({_q('d', source_id_col)} AS STRING)"
    source_id_label = source_id_col
else:
    source_id_expr = f"sha2(CONCAT_WS('||', CAST({_q('d', client_visit_guid_col)} AS STRING), CAST({code_expr} AS STRING), CAST({diagnosis_ts_expr} AS STRING)), 256)"
    source_id_label = "hash"

diag_type_expr = _q("d", diag_type_col) if diag_type_col else "NULL"
diag_comment_expr = _q("d", diag_comment_col) if diag_comment_col else "NULL"
if patient_source_col:
    person_source_value_expr = f"CONCAT_WS(CHR(31), '{source}', 'cv3client', 'GUID', CAST({_q('d', patient_source_col)} AS STRING))"
else:
    person_source_value_expr = "NULL"

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW silver_condition_occurrence AS
WITH source_data AS (
  SELECT
    {_q('d', client_visit_guid_col)} AS client_visit_guid,
    {diag_type_expr} AS diagnosis_type,
    {diag_comment_expr} AS diagnosis_comment,
    {person_source_value_expr} AS person_source_value,
    {diagnosis_ts_expr} AS diagnosis_ts,
    {code_expr} AS source_code,
    {vocab_expr} AS source_vocabulary_id,
    CONCAT_WS(
      CHR(31),
      '{source}',
      'vw_diagnosis',
      '{source_id_label}',
      {source_id_expr},
      'vocab',
      {vocab_expr},
      'code',
      {code_expr}
    ) AS condition_occurrence_source_value,
    CONCAT_WS(
      CHR(31),
      '{source}',
      'dbo_cv3clientvisit',
      'GUID',
      CAST({_q('d', client_visit_guid_col)} AS STRING)
    ) AS visit_occurrence_source_value
  {source_join_sql}
  WHERE {_q('d', client_visit_guid_col)} IS NOT NULL
), mapped AS (
  SELECT
    sd.condition_occurrence_source_value,
    COALESCE(stp.person_id, stvo.person_id, vo.person_id) AS person_id,
    COALESCE(std_concept.concept_id, src_standard_concept.concept_id, 0) AS condition_concept_id,
    DATE(COALESCE(sd.diagnosis_ts, vo.visit_start_datetime, vo.visit_start_date)) AS condition_start_date,
    CAST(COALESCE(sd.diagnosis_ts, vo.visit_start_datetime, vo.visit_start_date) AS TIMESTAMP) AS condition_start_datetime,
    CAST(NULL AS DATE) AS condition_end_date,
    CAST(NULL AS TIMESTAMP) AS condition_end_datetime,
    CAST(32817 AS BIGINT) AS condition_type_concept_id,
    CAST(0 AS BIGINT) AS condition_status_concept_id,
    CAST(NULL AS STRING) AS stop_reason,
    CAST(NULL AS BIGINT) AS provider_id,
    COALESCE(stvo.visit_occurrence_id, vo.visit_occurrence_id) AS visit_occurrence_id,
    CAST(NULL AS BIGINT) AS visit_detail_id,
    sd.visit_occurrence_source_value,
    sd.source_code AS condition_source_value,
    COALESCE(src_concept.concept_id, 0) AS condition_source_concept_id,
    CONCAT_WS(' | ', sd.diagnosis_type, sd.diagnosis_comment) AS condition_status_source_value,
    '{source}' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp,
    ROW_NUMBER() OVER (
      PARTITION BY sd.condition_occurrence_source_value
      ORDER BY COALESCE(sd.diagnosis_ts, vo.visit_start_datetime, vo.visit_start_date) DESC
    ) AS rn
  FROM source_data sd
  LEFT JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = sd.person_source_value
   AND stp.source_system = '{source}'
   AND stp.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = sd.visit_occurrence_source_value
   AND stvo.source_system = '{source}'
   AND stvo.active_flag = TRUE
  LEFT JOIN _exponent.omop_scm.visit_occurrence vo
    ON vo.visit_source_value = sd.visit_occurrence_source_value
  LEFT JOIN _exponent.omop.concept src_concept
    ON src_concept.concept_code = sd.source_code
   AND src_concept.vocabulary_id = sd.source_vocabulary_id
   AND src_concept.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept src_standard_concept
    ON src_standard_concept.concept_id = src_concept.concept_id
   AND src_standard_concept.standard_concept = 'S'
   AND src_standard_concept.domain_id = 'Condition'
   AND src_standard_concept.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept_relationship cr
    ON cr.concept_id_1 = src_concept.concept_id
   AND cr.relationship_id = 'Maps to'
   AND cr.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept std_concept
    ON std_concept.concept_id = cr.concept_id_2
   AND std_concept.standard_concept = 'S'
   AND std_concept.domain_id = 'Condition'
   AND std_concept.invalid_reason IS NULL
  WHERE sd.source_code IS NOT NULL
    AND sd.source_vocabulary_id IS NOT NULL
    AND COALESCE(stp.person_id, stvo.person_id, vo.person_id) IS NOT NULL
    AND COALESCE(sd.diagnosis_ts, vo.visit_start_datetime, vo.visit_start_date) IS NOT NULL
)
SELECT
  condition_occurrence_source_value,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  visit_occurrence_source_value,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value,
  source_system,
  last_mod_tsp
FROM mapped
WHERE rn = 1
""")

diagnostic_sql = f"""
WITH source_data AS (
  SELECT
    {_q('d', client_visit_guid_col)} AS client_visit_guid,
    {person_source_value_expr} AS person_source_value,
    {diagnosis_ts_expr} AS diagnosis_ts,
    {code_expr} AS source_code,
    {vocab_expr} AS source_vocabulary_id,
    CONCAT_WS(
      CHR(31),
      '{source}',
      'dbo_cv3clientvisit',
      'GUID',
      CAST({_q('d', client_visit_guid_col)} AS STRING)
    ) AS visit_occurrence_source_value
  {source_join_sql}
), diagnosed AS (
  SELECT
    sd.*,
    stp.person_id AS direct_person_id,
    stvo.person_id AS mapped_visit_person_id,
    stvo.visit_occurrence_id AS mapped_visit_occurrence_id,
    vo.person_id AS gold_visit_person_id,
    vo.visit_occurrence_id AS gold_visit_occurrence_id,
    vo.visit_start_datetime AS gold_visit_start_datetime,
    vo.visit_start_date AS gold_visit_start_date,
    src_concept.concept_id AS source_concept_id,
    COALESCE(std_concept.concept_id, src_standard_concept.concept_id) AS standard_condition_concept_id
  FROM source_data sd
  LEFT JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = sd.person_source_value
   AND stp.source_system = '{source}'
   AND stp.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = sd.visit_occurrence_source_value
   AND stvo.source_system = '{source}'
   AND stvo.active_flag = TRUE
  LEFT JOIN _exponent.omop_scm.visit_occurrence vo
    ON vo.visit_source_value = sd.visit_occurrence_source_value
  LEFT JOIN _exponent.omop.concept src_concept
    ON src_concept.concept_code = sd.source_code
   AND src_concept.vocabulary_id = sd.source_vocabulary_id
   AND src_concept.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept src_standard_concept
    ON src_standard_concept.concept_id = src_concept.concept_id
   AND src_standard_concept.standard_concept = 'S'
   AND src_standard_concept.domain_id = 'Condition'
   AND src_standard_concept.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept_relationship cr
    ON cr.concept_id_1 = src_concept.concept_id
   AND cr.relationship_id = 'Maps to'
   AND cr.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept std_concept
    ON std_concept.concept_id = cr.concept_id_2
   AND std_concept.standard_concept = 'S'
   AND std_concept.domain_id = 'Condition'
   AND std_concept.invalid_reason IS NULL
)
SELECT '01 raw vw_diagnosis rows' AS stage, COUNT(*) AS rows FROM source_data
UNION ALL SELECT '02 with ClientVisitGUID', COUNT(*) FROM source_data WHERE client_visit_guid IS NOT NULL
UNION ALL SELECT '03 with diagnosis code', COUNT(*) FROM source_data WHERE source_code IS NOT NULL AND source_vocabulary_id IS NOT NULL
UNION ALL SELECT '04 with direct person mapping', COUNT(*) FROM diagnosed WHERE direct_person_id IS NOT NULL
UNION ALL SELECT '05 with visit mapping', COUNT(*) FROM diagnosed WHERE mapped_visit_occurrence_id IS NOT NULL OR gold_visit_occurrence_id IS NOT NULL
UNION ALL SELECT '06 with any person mapping', COUNT(*) FROM diagnosed WHERE COALESCE(direct_person_id, mapped_visit_person_id, gold_visit_person_id) IS NOT NULL
UNION ALL SELECT '07 with usable date', COUNT(*) FROM diagnosed WHERE COALESCE(diagnosis_ts, gold_visit_start_datetime, gold_visit_start_date) IS NOT NULL
UNION ALL SELECT '08 source concept matched', COUNT(*) FROM diagnosed WHERE source_concept_id IS NOT NULL
UNION ALL SELECT '09 standard Condition concept matched', COUNT(*) FROM diagnosed WHERE standard_condition_concept_id IS NOT NULL
"""

print("SCM diagnosis staging drop-off summary")
display(spark.sql(diagnostic_sql))

silver_count = spark.table("silver_condition_occurrence").count()
print(f"Silver staging rows from vw_diagnosis: {silver_count:,}")
display(spark.table("silver_condition_occurrence").limit(25))




In [0]:
# -- Merge to Silver layer --
spark.sql("""
MERGE INTO _exponent.omop_silver.condition_occurrence AS t
USING silver_condition_occurrence AS s
ON t.condition_occurrence_source_value = s.condition_occurrence_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.condition_concept_id <=> s.condition_concept_id)
  OR NOT (t.condition_start_date <=> s.condition_start_date)
  OR NOT (t.condition_start_datetime <=> s.condition_start_datetime)
  OR NOT (t.condition_end_date <=> s.condition_end_date)
  OR NOT (t.condition_end_datetime <=> s.condition_end_datetime)
  OR NOT (t.condition_type_concept_id <=> s.condition_type_concept_id)
  OR NOT (t.condition_status_concept_id <=> s.condition_status_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.visit_detail_id <=> s.visit_detail_id)
  OR NOT (t.visit_occurrence_source_value <=> s.visit_occurrence_source_value)
  OR NOT (t.condition_source_value <=> s.condition_source_value)
  OR NOT (t.condition_source_concept_id <=> s.condition_source_concept_id)
  OR NOT (t.condition_status_source_value <=> s.condition_status_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                     = s.person_id,
  t.condition_concept_id          = s.condition_concept_id,
  t.condition_start_date          = s.condition_start_date,
  t.condition_start_datetime      = s.condition_start_datetime,
  t.condition_end_date            = s.condition_end_date,
  t.condition_end_datetime        = s.condition_end_datetime,
  t.condition_type_concept_id     = s.condition_type_concept_id,
  t.condition_status_concept_id   = s.condition_status_concept_id,
  t.stop_reason                   = s.stop_reason,
  t.provider_id                   = s.provider_id,
  t.visit_occurrence_id           = s.visit_occurrence_id,
  t.visit_detail_id               = s.visit_detail_id,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.condition_source_value        = s.condition_source_value,
  t.condition_source_concept_id   = s.condition_source_concept_id,
  t.condition_status_source_value = s.condition_status_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_source_value,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  visit_occurrence_source_value,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.condition_occurrence_source_value,
  s.person_id,
  s.condition_concept_id,
  s.condition_start_date,
  s.condition_start_datetime,
  s.condition_end_date,
  s.condition_end_datetime,
  s.condition_type_concept_id,
  s.condition_status_concept_id,
  s.stop_reason,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.visit_occurrence_source_value,
  s.condition_source_value,
  s.condition_source_concept_id,
  s.condition_status_source_value,
  s.source_system,
  CURRENT_TIMESTAMP()
);
""")


In [0]:
# -- Insert new mappings to source_to_condition_occurrence --
spark.sql("""
INSERT INTO _exponent.omop_mapping.source_to_condition_occurrence (
    source_system,
    condition_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.condition_occurrence_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, condition_occurrence_source_value, last_mod_tsp
    FROM _exponent.omop_silver.condition_occurrence
    WHERE source_system = 'allscripts_scm'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_condition_occurrence x
  ON s.condition_occurrence_source_value = x.condition_occurrence_source_value
 AND s.source_system = x.source_system;
""")


In [0]:
# -- Merge to Gold layer --
spark.sql("""
MERGE INTO _exponent.omop_scm.condition_occurrence AS gold
USING (
  SELECT
    sco.condition_occurrence_id,
    s.person_id,
    s.condition_concept_id,
    s.condition_start_date,
    s.condition_start_datetime,
    s.condition_end_date,
    s.condition_end_datetime,
    s.condition_type_concept_id,
    s.condition_status_concept_id,
    s.stop_reason,
    s.provider_id AS provider_id,
    COALESCE(s.visit_occurrence_id, stvo.visit_occurrence_id, vo.visit_occurrence_id) AS visit_occurrence_id,
    s.visit_detail_id AS visit_detail_id,
    s.condition_source_value,
    s.condition_source_concept_id,
    s.condition_status_source_value
  FROM _exponent.omop_silver.condition_occurrence s
  JOIN _exponent.omop_mapping.source_to_condition_occurrence sco
    ON sco.condition_occurrence_source_value = s.condition_occurrence_source_value
   AND sco.source_system = s.source_system
   AND sco.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = s.visit_occurrence_source_value
   AND stvo.source_system = s.source_system
   AND stvo.active_flag = TRUE
  LEFT JOIN _exponent.omop_scm.visit_occurrence vo
    ON vo.visit_source_value = s.visit_occurrence_source_value
  WHERE s.source_system = 'allscripts_scm'
    AND s.person_id IS NOT NULL
) AS src
ON gold.condition_occurrence_id = src.condition_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                     = src.person_id,
  gold.condition_concept_id          = src.condition_concept_id,
  gold.condition_start_date          = src.condition_start_date,
  gold.condition_start_datetime      = src.condition_start_datetime,
  gold.condition_end_date            = src.condition_end_date,
  gold.condition_end_datetime        = src.condition_end_datetime,
  gold.condition_type_concept_id     = src.condition_type_concept_id,
  gold.condition_status_concept_id   = src.condition_status_concept_id,
  gold.stop_reason                   = src.stop_reason,
  gold.provider_id                   = src.provider_id,
  gold.visit_occurrence_id           = src.visit_occurrence_id,
  gold.visit_detail_id               = src.visit_detail_id,
  gold.condition_source_value        = src.condition_source_value,
  gold.condition_source_concept_id   = src.condition_source_concept_id,
  gold.condition_status_source_value = src.condition_status_source_value

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_id,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value
)
VALUES (
  src.condition_occurrence_id,
  src.person_id,
  src.condition_concept_id,
  src.condition_start_date,
  src.condition_start_datetime,
  src.condition_end_date,
  src.condition_end_datetime,
  src.condition_type_concept_id,
  src.condition_status_concept_id,
  src.stop_reason,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.condition_source_value,
  src.condition_source_concept_id,
  src.condition_status_source_value
);
""")


In [0]:
# -- Merge to Gold layer --
spark.sql("""
MERGE INTO _exponent.omop_allscripts.condition_occurrence AS gold
USING (
  SELECT
    sco.condition_occurrence_id,
    s.person_id,
    s.condition_concept_id,
    s.condition_start_date,
    s.condition_start_datetime,
    s.condition_end_date,
    s.condition_end_datetime,
    s.condition_type_concept_id,
    s.condition_status_concept_id,
    s.stop_reason,
    s.provider_id AS provider_id,
    COALESCE(s.visit_occurrence_id, stvo.visit_occurrence_id, vo.visit_occurrence_id) AS visit_occurrence_id,
    s.visit_detail_id AS visit_detail_id,
    s.condition_source_value,
    s.condition_source_concept_id,
    s.condition_status_source_value
  FROM _exponent.omop_silver.condition_occurrence s
  JOIN _exponent.omop_mapping.source_to_condition_occurrence sco
    ON sco.condition_occurrence_source_value = s.condition_occurrence_source_value
   AND sco.source_system = s.source_system
   AND sco.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = s.visit_occurrence_source_value
   AND stvo.source_system = s.source_system
   AND stvo.active_flag = TRUE
  LEFT JOIN _exponent.omop_scm.visit_occurrence vo
    ON vo.visit_source_value = s.visit_occurrence_source_value
  WHERE s.source_system = 'allscripts_scm'
    AND s.person_id IS NOT NULL
) AS src
ON gold.condition_occurrence_id = src.condition_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                     = src.person_id,
  gold.condition_concept_id          = src.condition_concept_id,
  gold.condition_start_date          = src.condition_start_date,
  gold.condition_start_datetime      = src.condition_start_datetime,
  gold.condition_end_date            = src.condition_end_date,
  gold.condition_end_datetime        = src.condition_end_datetime,
  gold.condition_type_concept_id     = src.condition_type_concept_id,
  gold.condition_status_concept_id   = src.condition_status_concept_id,
  gold.stop_reason                   = src.stop_reason,
  gold.provider_id                   = src.provider_id,
  gold.visit_occurrence_id           = src.visit_occurrence_id,
  gold.visit_detail_id               = src.visit_detail_id,
  gold.condition_source_value        = src.condition_source_value,
  gold.condition_source_concept_id   = src.condition_source_concept_id,
  gold.condition_status_source_value = src.condition_status_source_value

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_id,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value
)
VALUES (
  src.condition_occurrence_id,
  src.person_id,
  src.condition_concept_id,
  src.condition_start_date,
  src.condition_start_datetime,
  src.condition_end_date,
  src.condition_end_datetime,
  src.condition_type_concept_id,
  src.condition_status_concept_id,
  src.stop_reason,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.condition_source_value,
  src.condition_source_concept_id,
  src.condition_status_source_value
);
""")


In [0]:
silver_rows = spark.sql("""
SELECT COUNT(*) AS silver_rows
FROM _exponent.omop_silver.condition_occurrence
WHERE source_system = 'allscripts_scm'
""")

gold_rows = spark.sql("""
SELECT COUNT(*) AS gold_rows
FROM _exponent.omop_scm.condition_occurrence
""")

print("SCM condition_occurrence counts after load")
display(silver_rows)
display(gold_rows)
display(
    spark.sql("""
    SELECT *
    FROM _exponent.omop_silver.condition_occurrence
    WHERE source_system = 'allscripts_scm'
    LIMIT 25
    """)
)
